# Apache Spark — Hands-on Lab & Answer Key
### RDD • PySpark DataFrame • Spark SQL

**Audience:** Beginner Spark learners  
**Format:** Each question is a Markdown cell followed by an executable solution code cell.  
**Duration:** ~2–3 hours (guided) or longer as an individual assignment.

The lab follows the topics in the supplied `spark_basics.py`: RDD transformations/actions, pair RDDs, joins, caching; DataFrame selection, filtering, computed columns, aggregations, windows, joins and null/dedup; Spark SQL WHERE, GROUP BY/HAVING, JOIN, WINDOW, CASE and CTE.

**Instructions:** Run the setup and data cells first. For each task, try your own code *before* looking at the next solution. Answers use only the sample data built in this notebook, so no external input files are required. Spark/PySpark must be installed.

## 0 — Environment setup

In [149]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("Spark_HandsOn_Assignment")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)

Spark version: 4.0.4
Default parallelism: 2


## 0.1 — Create the sample data
The **RDD sales dataset** is `(City, Category, Amount)`. The **DataFrame employee dataset** follows your original fields `(EmployeeID, Name, Department, Salary, City)`; extra records deliberately include duplicates and null values to practise cleaning.

In [150]:
numbers_rdd = sc.parallelize(list(range(1, 11)), numSlices=4)

sales_data = [
    ("Cairo", "Electronics", 1200),
    ("Cairo", "Furniture", 800),
    ("Dubai", "Electronics", 2200),
    ("Dubai", "Furniture", 1700),
    ("Riyadh", "Electronics", 1500),
    ("Cairo", "Electronics", 500),
    ("Riyadh", "Furniture", 700),
    ("Dubai", "Electronics", 600),
]
sales_rdd = sc.parallelize(sales_data, numSlices=3)
region_rdd = sc.parallelize([
    ("Cairo", "EG"), ("Dubai", "UAE"), ("Riyadh", "KSA")
])

employee_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", DoubleType(), True),
    StructField("City", StringType(), True),
])
employee_data = [
    (1, "Ahmed", "Engineering", 15000.0, "Cairo"),
    (2, "Sara", "Finance", 12000.0, "Dubai"),
    (3, "Mostafa", "Engineering", 18000.0, "Cairo"),
    (4, "Laila", "HR", 9000.0, "Riyadh"),
    (5, "Omar", "Finance", 13500.0, "Dubai"),
    (6, "Nour", "Engineering", 16500.0, "Riyadh"),
    (7, "Mona", "HR", None, "Cairo"),
    (8, "Youssef", "Finance", 14000.0, None),
]
employees_df = spark.createDataFrame(employee_data, employee_schema)
city_region_df = spark.createDataFrame([
    ("Cairo", "EG"), ("Dubai", "UAE"), ("Riyadh", "KSA")
], ["City", "Region"])

employees_df.createOrReplaceTempView("employees")
city_region_df.createOrReplaceTempView("city_region")

print("Numbers partitions:", numbers_rdd.getNumPartitions())
print("Sales rows:", sales_rdd.count())
employees_df.show(truncate=False)
city_region_df.show()

Numbers partitions: 4
Sales rows: 8
+----------+-------+-----------+-------+------+
|EmployeeID|Name   |Department |Salary |City  |
+----------+-------+-----------+-------+------+
|1         |Ahmed  |Engineering|15000.0|Cairo |
|2         |Sara   |Finance    |12000.0|Dubai |
|3         |Mostafa|Engineering|18000.0|Cairo |
|4         |Laila  |HR         |9000.0 |Riyadh|
|5         |Omar   |Finance    |13500.0|Dubai |
|6         |Nour   |Engineering|16500.0|Riyadh|
|7         |Mona   |HR         |NULL   |Cairo |
|8         |Youssef|Finance    |14000.0|NULL  |
+----------+-------+-----------+-------+------+

+------+------+
|  City|Region|
+------+------+
| Cairo|    EG|
| Dubai|   UAE|
|Riyadh|   KSA|
+------+------+



---
# Part 1 — RDD (30 points)
Each question is worth **5 points**. Practise transformations vs actions, `map`, `filter`, `flatMap`, pair RDD grouping, sorting, and joins.

### RDD.1 — map + filter

**Task:** Using `numbers_rdd`, multiply all numbers by **3**, then keep only those **greater than 15**. Display the result with `collect()`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [151]:
# Write your solution here
numbers_filtered=numbers_rdd.map(lambda x : x*3)\
.filter(lambda x : x > 15)



In [152]:
numbers_filtered.collect()

[18, 21, 24, 27, 30]

### RDD.2 — flatMap word split

**Task:** Create an RDD from `["hello spark", "spark rdd basics", "hello data"]`. Use `flatMap()` to produce **one word per element**, and compare it with `map()`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [153]:
# Write your solution here
rdd_string_list = sc.parallelize(["hello spark", "spark rdd basics", "hello data"])
rdd_flatmap = rdd_string_list.flatMap(lambda i : i.split(" "))
rdd_map = rdd_string_list.map(lambda i : i.split(" "))

In [154]:
rdd_flatmap.collect()
# Flattens the results into a single RDD, with each word as a separate element.

['hello', 'spark', 'spark', 'rdd', 'basics', 'hello', 'data']

In [155]:
rdd_map.collect()
# Keeps each mapped result as a single element, even if it contains multiple words.

[['hello', 'spark'], ['spark', 'rdd', 'basics'], ['hello', 'data']]

### RDD.3 — RDD actions

**Task:** For `numbers_rdd`, calculate: total elements with `count()`, first three elements using `take(3)`, and sum using `reduce()`. Explain why `collect()` should be used cautiously for large data.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [156]:
# Write your solution here
numbers_rdd.count()


10

In [157]:
numbers_rdd.take(3)

[1, 2, 3]

In [158]:
rdd_reduce= numbers_rdd.reduce(lambda a, b : a+b)
rdd_reduce


55

In [159]:
# collect() should be used cautiously for large datasets because it returns all elements to the Driver, which can consume a large amount of memory and cause an OutOfMemoryError.

### RDD.4 — reduceByKey and sortBy

**Task:** Calculate **total sales amount per city** using a pair RDD and `reduceByKey()`, sorted from highest to lowest total.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [160]:
# Write your solution here
rdd_pairs=sales_rdd.map(lambda row : (row[0] , row[2]))
rdd_total_sales=rdd_pairs.reduceByKey(lambda a,b : a+b)
city_totals=rdd_total_sales.sortBy(lambda x : x[1], ascending=False)
city_totals.collect()



[('Dubai', 4500), ('Cairo', 2500), ('Riyadh', 2200)]

### RDD.5 — groupByKey and join

**Task:** Display **all individual sales amounts** for each city using `groupByKey()`. Then join **city totals** to `region_rdd` to show `(City, (Total, Region))`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [161]:
# Write your solution here
rdd_pairs = sales_rdd.map(lambda row: (row[0], row[2]))
rdd_grouped_by_city = rdd_pairs.groupByKey().mapValues(list)
rdd_grouped_by_city.collect()


[('Dubai', [2200, 1700, 600]),
 ('Cairo', [1200, 800, 500]),
 ('Riyadh', [1500, 700])]

In [162]:
rdd_city_totals=rdd_pairs.reduceByKey(lambda a, b : a+b)
rdd_joined = rdd_city_totals.join(region_rdd)
rdd_joined.collect()

[('Riyadh', (2200, 'KSA')), ('Cairo', (2500, 'EG')), ('Dubai', (4500, 'UAE'))]

### RDD.6 — Partitions and cache

**Task:** Show the number of partitions in `numbers_rdd`, cache `city_totals`, and execute **two actions** (`count` and `collect`) on the cached RDD.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [163]:
# Write your solution here
numbers_of_partitions = numbers_rdd.getNumPartitions()
print(numbers_of_partitions)



4


In [164]:
city_totals.cache()
city_totals.count()
city_totals.collect()

[('Dubai', 4500), ('Cairo', 2500), ('Riyadh', 2200)]

---
# Part 2 — PySpark DataFrame (35 points)
Seven questions, **5 points each**. Use the `employees_df` and `city_region_df` datasets.

### DF.1 — Schema, select and filter

**Task:** Print the employee schema. Select `Name`, `Department`, and `Salary`, keeping employees with non-null salary **greater than 13,000**.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [165]:
# Write your solution here
employees_df.printSchema()



root
 |-- EmployeeID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- City: string (nullable = true)



In [166]:
Employee_filter = employees_df.select("Name" , "Department" , "Salary")\
.filter(F.col("Salary") > 13000)
Employee_filter.show()

+-------+-----------+-------+
|   Name| Department| Salary|
+-------+-----------+-------+
|  Ahmed|Engineering|15000.0|
|Mostafa|Engineering|18000.0|
|   Omar|    Finance|13500.0|
|   Nour|Engineering|16500.0|
|Youssef|    Finance|14000.0|
+-------+-----------+-------+



### DF.2 — withColumn bonus

**Task:** Create `Bonus` equal to **10% of Salary**, and `TotalPay` equal to salary plus bonus. Display the new columns; preserve null values.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [167]:
# Write your solution here
df_totalpay=employees_df.withColumn("Bonus" , F.col("Salary") * .10)\
.withColumn("TotalPay",F.col("Salary")+F.col("Bonus"))
df_totalpay.show()


+----------+-------+-----------+-------+------+------+--------+
|EmployeeID|   Name| Department| Salary|  City| Bonus|TotalPay|
+----------+-------+-----------+-------+------+------+--------+
|         1|  Ahmed|Engineering|15000.0| Cairo|1500.0| 16500.0|
|         2|   Sara|    Finance|12000.0| Dubai|1200.0| 13200.0|
|         3|Mostafa|Engineering|18000.0| Cairo|1800.0| 19800.0|
|         4|  Laila|         HR| 9000.0|Riyadh| 900.0|  9900.0|
|         5|   Omar|    Finance|13500.0| Dubai|1350.0| 14850.0|
|         6|   Nour|Engineering|16500.0|Riyadh|1650.0| 18150.0|
|         7|   Mona|         HR|   NULL| Cairo|  NULL|    NULL|
|         8|Youssef|    Finance|14000.0|  NULL|1400.0| 15400.0|
+----------+-------+-----------+-------+------+------+--------+



### DF.3 — GroupBy aggregation

**Task:** For every department calculate `EmployeeCount`, `AvgSalary`, `MaxSalary`, and `MinSalary`. Sort by descending average salary. Remember that `avg` ignores null salaries.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [168]:
# Write your solution here
df_dep_summary = (
    employees_df.groupBy("Department")
    .agg(
          F.count("*").alias("EmployeeCount"),
          F.avg("Salary").alias("AvgSalary"),
          F.max("Salary").alias("MaxSalary"),
          F.min("Salary").alias("MinSalary")
        ).orderBy(F.col("AvgSalary").desc())
)
df_dep_summary.show()


+-----------+-------------+------------------+---------+---------+
| Department|EmployeeCount|         AvgSalary|MaxSalary|MinSalary|
+-----------+-------------+------------------+---------+---------+
|Engineering|            3|           16500.0|  18000.0|  15000.0|
|    Finance|            3|13166.666666666666|  14000.0|  12000.0|
|         HR|            2|            9000.0|   9000.0|   9000.0|
+-----------+-------------+------------------+---------+---------+



### DF.4 — Window rank and running average

**Task:** Within each department, rank employees from highest to lowest salary using `rank()`, and calculate a running salary average ordered by descending salary. Treat null salary as last.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [169]:
# Write your solution here
window_spec=Window.partitionBy("Department").orderBy(F.col("Salary").desc_nulls_last())
df_ranked=employees_df.withColumn("RankInDepth",F.rank().over(window_spec))\
.withColumn("RunningAvg" , F.avg("Salary").over(window_spec.rowsBetween(Window.unboundedPreceding,Window.currentRow)))
df_ranked.show()


+----------+-------+-----------+-------+------+-----------+------------------+
|EmployeeID|   Name| Department| Salary|  City|RankInDepth|        RunningAvg|
+----------+-------+-----------+-------+------+-----------+------------------+
|         3|Mostafa|Engineering|18000.0| Cairo|          1|           18000.0|
|         6|   Nour|Engineering|16500.0|Riyadh|          2|           17250.0|
|         1|  Ahmed|Engineering|15000.0| Cairo|          3|           16500.0|
|         8|Youssef|    Finance|14000.0|  NULL|          1|           14000.0|
|         5|   Omar|    Finance|13500.0| Dubai|          2|           13750.0|
|         2|   Sara|    Finance|12000.0| Dubai|          3|13166.666666666666|
|         4|  Laila|         HR| 9000.0|Riyadh|          1|            9000.0|
|         7|   Mona|         HR|   NULL| Cairo|          2|            9000.0|
+----------+-------+-----------+-------+------+-----------+------------------+



### DF.5 — Left join city-region

**Task:** Left join `employees_df` to `city_region_df` on `City`, retaining all employees even when their City is missing.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [170]:
# Write your solution here
df_joined = employees_df.join(city_region_df , on = "City", how="left")
df_joined.show()


+------+----------+-------+-----------+-------+------+
|  City|EmployeeID|   Name| Department| Salary|Region|
+------+----------+-------+-----------+-------+------+
| Cairo|         1|  Ahmed|Engineering|15000.0|    EG|
| Cairo|         3|Mostafa|Engineering|18000.0|    EG|
|Riyadh|         4|  Laila|         HR| 9000.0|   KSA|
| Dubai|         2|   Sara|    Finance|12000.0|   UAE|
| Cairo|         7|   Mona|         HR|   NULL|    EG|
|  NULL|         8|Youssef|    Finance|14000.0|  NULL|
|Riyadh|         6|   Nour|Engineering|16500.0|   KSA|
| Dubai|         5|   Omar|    Finance|13500.0|   UAE|
+------+----------+-------+-----------+-------+------+



### DF.6 — Null handling

**Task:** Replace missing salaries with **0.0** and missing cities with `"Unknown"`. Count null salary records **before** and **after** filling.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [171]:
# Before filling
employees_df.filter(F.col("Salary").isNull()).count()

1

In [172]:
# Before filling
employees_df.filter(F.col("City").isNull()).count()

1

In [173]:
# Write your solution here
df_filled= employees_df.fillna({"Salary":0.0,"City":"Unknown"})
df_filled.show()



+----------+-------+-----------+-------+-------+
|EmployeeID|   Name| Department| Salary|   City|
+----------+-------+-----------+-------+-------+
|         1|  Ahmed|Engineering|15000.0|  Cairo|
|         2|   Sara|    Finance|12000.0|  Dubai|
|         3|Mostafa|Engineering|18000.0|  Cairo|
|         4|  Laila|         HR| 9000.0| Riyadh|
|         5|   Omar|    Finance|13500.0|  Dubai|
|         6|   Nour|Engineering|16500.0| Riyadh|
|         7|   Mona|         HR|    0.0|  Cairo|
|         8|Youssef|    Finance|14000.0|Unknown|
+----------+-------+-----------+-------+-------+



In [174]:
#Afterfilling
df_filled.filter(F.col("Salary").isNull()).count()

0

In [175]:
#Afterfilling
df_filled.filter(F.col("City").isNull()).count()

0

### DF.7 — dropDuplicates vs distinct

**Task:** Create a small DataFrame with repeated rows. Show the difference between `distinct()` (whole row) and `dropDuplicates(["Department"])` (one record per department; other values are not deterministically selected).

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [176]:
# Write your solution here
data= [
    ("Ahmed", "IT", 12000),
    ("Mona", "HR", 9000),
    ("Ahmed", "IT", 12000),
    ("Ali", "IT", 15000),
    ("Mona", "HR", 9000)
]
columns = ["Name", "Department", "Salary"]
data=spark.createDataFrame(data,columns)
df_distinct = data.distinct()
df_distinct.show()

+-----+----------+------+
| Name|Department|Salary|
+-----+----------+------+
|Ahmed|        IT| 12000|
| Mona|        HR|  9000|
|  Ali|        IT| 15000|
+-----+----------+------+



In [177]:
df_duplicated =data.drop_duplicates(["Department"])
df_duplicated.show()

+-----+----------+------+
| Name|Department|Salary|
+-----+----------+------+
| Mona|        HR|  9000|
|Ahmed|        IT| 12000|
+-----+----------+------+



---
# Part 3 — Spark SQL (35 points)
Seven questions, **5 points each**. Answer using `spark.sql()` over the temporary views `employees` and `city_region`.

### SQL.1 — SELECT / WHERE / ORDER BY

**Task:** Return the `Name`, `Department`, and `Salary` of employees earning **more than 13,000**, highest salary first.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [178]:
# Write your solution here
spark.sql("""
select Name , Department , Salary From employees
where Salary > 13000
order by Salary desc
""").show()


+-------+-----------+-------+
|   Name| Department| Salary|
+-------+-----------+-------+
|Mostafa|Engineering|18000.0|
|   Nour|Engineering|16500.0|
|  Ahmed|Engineering|15000.0|
|Youssef|    Finance|14000.0|
|   Omar|    Finance|13500.0|
+-------+-----------+-------+



### SQL.2 — GROUP BY + HAVING

**Task:** For each department with **at least 2 employees**, show the number of employees and their average **non-null** salary (rounded to 2 decimals). Sort by average salary descending.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [179]:
# Write your solution here
spark.sql("""
select  Department  , count(*) as Emp_count , round(AVG(Salary),2) as Avg_Salary From Employees
group by Department
having count(*) >= 2
order by Avg_Salary desc
""").show()

+-----------+---------+----------+
| Department|Emp_count|Avg_Salary|
+-----------+---------+----------+
|Engineering|        3|   16500.0|
|    Finance|        3|  13166.67|
|         HR|        2|    9000.0|
+-----------+---------+----------+



### SQL.3 — LEFT JOIN

**Task:** Return `Name`, `City`, and `Region` for **all employees** using a LEFT JOIN between `employees` and `city_region`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [180]:
# Write your solution here
spark.sql("""
select e.Name , e.City , c.Region from employees e
left join city_region  c on c.City = e.City
""").show()

+-------+------+------+
|   Name|  City|Region|
+-------+------+------+
|  Ahmed| Cairo|    EG|
|Mostafa| Cairo|    EG|
|  Laila|Riyadh|   KSA|
|   Sara| Dubai|   UAE|
|   Mona| Cairo|    EG|
|Youssef|  NULL|  NULL|
|   Nour|Riyadh|   KSA|
|   Omar| Dubai|   UAE|
+-------+------+------+



### SQL.4 — Window RANK and AVG

**Task:** Return `Name`, `Department`, `Salary`, a salary `RANK()` within each department, and each department’s average salary.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [181]:
# Write your solution here
spark.sql("""
select
 Name,
 Department ,
 Salary ,
 RANK() over(partition by  Department order by Salary Desc  ) as salary_rank ,
 round(avg(salary) over(PARTITION BY Department),2) as Dept_Avg_Salary
 from employees
""").show()


+-------+-----------+-------+-----------+---------------+
|   Name| Department| Salary|salary_rank|Dept_Avg_Salary|
+-------+-----------+-------+-----------+---------------+
|Mostafa|Engineering|18000.0|          1|        16500.0|
|   Nour|Engineering|16500.0|          2|        16500.0|
|  Ahmed|Engineering|15000.0|          3|        16500.0|
|Youssef|    Finance|14000.0|          1|       13166.67|
|   Omar|    Finance|13500.0|          2|       13166.67|
|   Sara|    Finance|12000.0|          3|       13166.67|
|  Laila|         HR| 9000.0|          1|         9000.0|
|   Mona|         HR|   NULL|          2|         9000.0|
+-------+-----------+-------+-----------+---------------+



### SQL.5 — CASE WHEN salary band

**Task:** Create `SalaryBand` with these rules: salary **>= 16000 → High**, **>= 12000 → Medium**, otherwise **Low**; missing salary → **Unknown**.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [182]:
# Write your solution here
spark.sql("""
select Name , Department , Salary ,
Case
WHEN Salary IS NULL THEN "Unknown"
when Salary >= 16000 then "High"
when Salary >= 12000 then "Medium"
else "Low"
end
as SalaryBand
from employees
""").show()


+-------+-----------+-------+----------+
|   Name| Department| Salary|SalaryBand|
+-------+-----------+-------+----------+
|  Ahmed|Engineering|15000.0|    Medium|
|   Sara|    Finance|12000.0|    Medium|
|Mostafa|Engineering|18000.0|      High|
|  Laila|         HR| 9000.0|       Low|
|   Omar|    Finance|13500.0|    Medium|
|   Nour|Engineering|16500.0|      High|
|   Mona|         HR|   NULL|   Unknown|
|Youssef|    Finance|14000.0|    Medium|
+-------+-----------+-------+----------+



### SQL.6 — CTE and above-department-average

**Task:** Using a `WITH` CTE, calculate department average salary, then return employees with salary **strictly greater** than their own department average. Include the amount above average.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [183]:
# Write your solution here
spark.sql("""with CTE AS
(
    select department , avg(Salary) as AvgSalary
    from employees
    group by department

)
        select * from CTE
        WHERE AvgSalary > (
        SELECT AVG(Salary)
        FROM employees )
""").show()

+-----------+---------+
| department|AvgSalary|
+-----------+---------+
|Engineering|  16500.0|
+-----------+---------+



### SQL.7 — CTE + JOIN + HAVING challenge

**Task:** Write a SQL query to find the **total payroll per region**, excluding employees with missing salaries or unknown region. Return only regions with total payroll **greater than 20,000**.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [184]:
# Write your solution here
spark.sql("""
with cte as (
select c.region , sum(e.salary) as totalPayroll
from employees e
join city_region c
on e.city = c.city
WHERE e.Salary IS NOT NULL
AND c.Region IS NOT NULL
GROUP BY c.Region

)
select * from cte
where TotalPayroll > 20000
order by totalPayroll desc
""").show()


+------+------------+
|region|totalPayroll|
+------+------------+
|    EG|     33000.0|
|   UAE|     25500.0|
|   KSA|     25500.0|
+------+------------+



---
## Wrap-up and grading rubric

| Section | Questions | Marks |
|---|---:|---:|
| RDD | 6 | 30 |
| PySpark DataFrame | 7 | 35 |
| Spark SQL | 7 | 35 |
| **Total** | **20** | **100** |

**Suggested grading for each task (5 marks):** 3 for correct transformations/query logic, 1 for correct output, 1 for appropriate Spark API usage.  
**Discussion:** Why does `reduceByKey` generally scale better than `groupByKey` for sums? What triggers lazy execution? Why is `collect` risky? How do Spark SQL and DataFrame APIs relate?

**Optional exercise:** Implement one city-level sales summary in **all three APIs** and compare results.

In [185]:
#reduceByKey: Aggregates values with the same key using a function, performing local aggregation to reduce shuffle data.
#groupByKey: Groups all values with the same key into an iterable collection without local aggregation.
#Lazy Execution: Spark delays executing transformations until an action triggers the computation (Filter , Select , etc ).
#collect() is risky because it brings all data to the Driver, which may cause out-of-memory errors when the dataset is large.
#Spark SQL and DataFrame APIs use the same Spark SQL engine and Catalyst optimizer to execute queries efficiently.



In [186]:
# Optional: compare the same sales totals in all three APIs
rdd_result = dict(sales_rdd.map(lambda r: (r[0], r[2])).reduceByKey(lambda a,b: a+b).collect())
sales_df = spark.createDataFrame(sales_data, ["City", "Category", "Amount"])
df_result = {r["City"]: r["Total"] for r in sales_df.groupBy("City").agg(F.sum("Amount").alias("Total")).collect()}
sales_df.createOrReplaceTempView("sales")
sql_result = {r["City"]: r["Total"] for r in spark.sql("SELECT City, SUM(Amount) AS Total FROM sales GROUP BY City").collect()}
print("RDD:", rdd_result)
print("DataFrame:", df_result)
print("SQL:", sql_result)
assert rdd_result == df_result == sql_result
print("✅ All three APIs give the same totals")

RDD: {'Dubai': 4500, 'Cairo': 2500, 'Riyadh': 2200}
DataFrame: {'Cairo': 2500, 'Dubai': 4500, 'Riyadh': 2200}
SQL: {'Cairo': 2500, 'Dubai': 4500, 'Riyadh': 2200}
✅ All three APIs give the same totals


**End of lab.** Keep the Spark session alive if you want to explore the data further. Run `spark.stop()` when finished.